# Датасет

In [1]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", "split")

In [2]:
ds['train'][0]

{'text': 'i didnt feel humiliated', 'label': 0}

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [4]:
num_labels = len(set(ds["train"]["label"]))

# Модель

In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased", 
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

In [8]:
tokenized_ds = ds.map(tokenize_function, batched=True)

# Метрики до дообучения

In [10]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

In [19]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../data/models/bert_base_uncased",
    eval_strategy="epoch",
    num_train_epochs=0,  # Не обучаем модель
    per_device_eval_batch_size=16,
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics,
)

In [22]:
results = trainer.evaluate()

In [25]:
print(f"Loss: {results['eval_loss']:.4f}")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1: {results['eval_f1']:.4f}")

Loss: 1.7824
Accuracy: 0.2715
F1: 0.1188


In [26]:
import torch
from sklearn.metrics import classification_report

test_dataloader = trainer.get_test_dataloader(tokenized_ds["test"])
predictions = []
labels = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(trainer.model.device) for k, v in batch.items() if k != 'labels'}
        outputs = trainer.model(**batch)
        predictions.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        
true_labels = tokenized_ds["test"]["label"]
print("\Отчет о классификации на test:")
print(classification_report(true_labels, predictions))

\Отчет о классификации на test:
              precision    recall  f1-score   support

           0       0.29      0.98      0.44       581
           1       0.00      0.00      0.00       695
           2       0.00      0.00      0.00       159
           3       0.00      0.00      0.00       275
           4       0.00      0.00      0.00       224
           5       0.00      0.00      0.00        66

    accuracy                           0.29      2000
   macro avg       0.05      0.16      0.07      2000
weighted avg       0.08      0.29      0.13      2000



/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 

- все предикты модели: 0 - sadness

# Full finetuning

In [31]:
# Посмотрим количество размороженных параметров для дообучения
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"Замороженных параметров: {frozen_params:,} ({frozen_params/total_params:.2%})")

Всего параметров: 109,486,854
Обучаемых параметров: 109,486,854 (100.00%)
Замороженных параметров: 0 (0.00%)


In [32]:
training_args = TrainingArguments(
    output_dir="../data/models/bert_base_uncased",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics,
)

- обучал в colab

In [ ]:
trainer.train()
# Epoch	Training Loss	Validation Loss	Accuracy	F1
# 1        	0.229000	0.193179	0.929500	0.930128
# 2        	0.132900	0.161190	0.939500	0.939274
# 3        	0.087800	0.160541	0.940000	0.939858

- время дообучения: 1:15:24

- ресурсы:

In [ ]:
# Sat Apr 26 12:26:43 2025       
# +-----------------------------------------------------------------------------------------+
# | NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
# |-----------------------------------------+------------------------+----------------------+
# | GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
# | Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
# |                                         |                        |               MIG M. |
# |=========================================+========================+======================|
# |   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
# | N/A   71C    P0             30W /   70W |    6492MiB /  15360MiB |      0%      Default |
# |                                         |                        |                  N/A |
# +-----------------------------------------+------------------------+----------------------+
                                                                                         
# +-----------------------------------------------------------------------------------------+
# | Processes:                                                                              |
# |  GPU   GI   CI        PID   Type   Process name                              GPU Memory |
# |        ID   ID                                                               Usage      |
# |=========================================================================================|
# +-----------------------------------------------------------------------------------------+

In [ ]:
results = trainer.evaluate()

In [ ]:
print(f"Loss: {results['eval_loss']:.4f}")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1: {results['eval_f1']:.4f}")
# Loss: 0.1605
# Accuracy: 0.9400
# F1: 0.9399

In [ ]:
import torch
from sklearn.metrics import classification_report

test_dataloader = trainer.get_test_dataloader(tokenized_ds["test"])
predictions = []
labels = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(trainer.model.device) for k, v in batch.items() if k != 'labels'}
        outputs = trainer.model(**batch)
        predictions.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        
true_labels = tokenized_ds["test"]["label"]
print("\Отчет о классификации на test:")
print(classification_report(true_labels, predictions))

# \Отчет о классификации на test:
#               precision    recall  f1-score   support

#            0       0.96      0.97      0.96       581
#            1       0.95      0.95      0.95       695
#            2       0.86      0.82      0.84       159
#            3       0.93      0.91      0.92       275
#            4       0.87      0.91      0.89       224
#            5       0.78      0.68      0.73        66

#     accuracy                           0.93      2000
#    macro avg       0.89      0.87      0.88      2000
# weighted avg       0.93      0.93      0.93      2000

# linear probing

Архитектура: Сложная многослойная голова (LayerNorm -> Dropout -> Linear -> ReLU -> ...) выбрана для увеличения выразительной способности при фиксированных весах BERT

In [53]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [54]:
import torch.nn as nn

model.classifier = nn.Sequential(
    nn.LayerNorm(model.config.hidden_size),
    nn.Dropout(0.1),
    nn.Linear(model.config.hidden_size, 512),
    nn.ReLU(),
    nn.LayerNorm(512),
    nn.Dropout(0.2),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.LayerNorm(256),
    nn.Dropout(0.1),
    nn.Linear(256, num_labels)
)

In [55]:
# Заморозим веса базовой модели
for param_name, param in model.named_parameters():
    if 'classifier' not in param_name:
        param.requires_grad = False

In [56]:
# Посмотрим количество размороженных параметров для дообучения
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"Замороженных параметров: {frozen_params:,} ({frozen_params/total_params:.2%})")

Всего параметров: 110,011,910
Обучаемых параметров: 529,670 (0.48%)
Замороженных параметров: 109,482,240 (99.52%)


In [57]:
training_args = TrainingArguments(
    output_dir="../data/models/bert_base_uncased",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [58]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics,
)

In [60]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.605100,1.550479,0.412000,0.317143
2,1.570300,1.536971,0.406500,0.307905
3,1.563500,1.530792,0.422000,0.321933


TrainOutput(global_step=1500, training_loss=1.5796581217447916, metrics={'train_runtime': 2997.0379, 'train_samples_per_second': 16.016, 'train_steps_per_second': 0.5, 'total_flos': 1.308842309615616e+16, 'train_loss': 1.5796581217447916, 'epoch': 3.0})

- время дообучения: 49:54

- ресурсы: обучение на CPU

In [61]:
results = trainer.evaluate()

In [62]:
print(f"Loss: {results['eval_loss']:.4f}")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1: {results['eval_f1']:.4f}")

Loss: 1.5308
Accuracy: 0.4220
F1: 0.3219


In [63]:
import torch
from sklearn.metrics import classification_report

test_dataloader = trainer.get_test_dataloader(tokenized_ds["test"])
predictions = []
labels = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(trainer.model.device) for k, v in batch.items() if k != 'labels'}
        outputs = trainer.model(**batch)
        predictions.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        
true_labels = tokenized_ds["test"]["label"]
print("\Отчет о классификации на test:")
print(classification_report(true_labels, predictions))

\Отчет о классификации на test:
              precision    recall  f1-score   support

           0       0.39      0.48      0.43       581
           1       0.43      0.80      0.56       695
           2       0.00      0.00      0.00       159
           3       0.00      0.00      0.00       275
           4       0.00      0.00      0.00       224
           5       0.00      0.00      0.00        66

    accuracy                           0.42      2000
   macro avg       0.14      0.21      0.16      2000
weighted avg       0.26      0.42      0.32      2000



/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 

# LoRA

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, PeftModel, PeftConfig

# Настроим конфигурацию LoRA
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,         # Задача классификации последовательностей
    inference_mode=False,               
    r=8,                                # Ранг адаптера (компромисс между производительностью и количеством параметров)
    lora_alpha=32,                      # Параметр масштабирования для LoRA
    lora_dropout=0.1,                   # Вероятность дропаута для регуляризации
    target_modules=["query", "key", "value"], # Модули для применения LoRA (самовнимание в BERT)
)

In [ ]:
model = get_peft_model(model, peft_config)

In [ ]:
# Посмотрим количество размороженных параметров для дообучения
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"Замороженных параметров: {frozen_params:,} ({frozen_params/total_params:.2%})")

Всего параметров: 109,933,836
Обучаемых параметров: 446,982 (0.41%)
Замороженных параметров: 109,486,854 (99.59%)


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../data/models/bert_base_LoRA",
    learning_rate=1e-4,                 # Немного более высокая скорость обучения для LoRA
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics,
)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
# обучал в colab
trainer.train()
# Epoch	Training Loss	Validation Loss	Accuracy	F1
# 1	1.254100	0.865412	0.705000	0.657505
# 2	0.726100	0.534992	0.820500	0.814304
# 3	0.542800	0.467080	0.843500	0.841267

- время дообучения: 1:02:30

- ресурсы:

In [ ]:
# +-----------------------------------------------------------------------------------------+
# | NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
# |-----------------------------------------+------------------------+----------------------+
# | GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
# | Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
# |                                         |                        |               MIG M. |
# |=========================================+========================+======================|
# |   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
# | N/A   77C    P0             69W /   70W |    9478MiB /  15360MiB |    100%      Default |
# |                                         |                        |                  N/A |
# +-----------------------------------------+------------------------+----------------------+
                                                                                         
# +-----------------------------------------------------------------------------------------+
# | Processes:                                                                              |
# |  GPU   GI   CI        PID   Type   Process name                              GPU Memory |
# |        ID   ID                                                               Usage      |
# |=========================================================================================|
# +-----------------------------------------------------------------------------------------+

In [ ]:
results = trainer.evaluate()

In [ ]:
print(f"Loss: {results['eval_loss']:.4f}")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1: {results['eval_f1']:.4f}")
# Loss: 0.4671
# Accuracy: 0.8435
# F1: 0.8413

In [ ]:
import torch
from sklearn.metrics import classification_report

test_dataloader = trainer.get_test_dataloader(tokenized_ds["test"])
predictions = []
labels = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(trainer.model.device) for k, v in batch.items() if k != 'labels'}
        outputs = trainer.model(**batch)
        predictions.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        
true_labels = tokenized_ds["test"]["label"]
print("\Отчет о классификации на test:")
print(classification_report(true_labels, predictions))

#               precision    recall  f1-score   support

#            0       0.88      0.89      0.88       581
#            1       0.90      0.92      0.91       695
#            2       0.72      0.64      0.68       159
#            3       0.79      0.80      0.79       275
#            4       0.79      0.84      0.81       224
#            5       0.83      0.44      0.57        66

#     accuracy                           0.85      2000
#    macro avg       0.82      0.76      0.78      2000
# weighted avg       0.85      0.85      0.85      2000

# Prefix finetuning

In [78]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=num_labels
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [79]:
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,    
    )

In [80]:
tokenized_ds = ds.map(tokenize_fn, batched=True)

In [81]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer)

In [82]:
from peft import PrefixTuningConfig, get_peft_model, TaskType, PeftModel, PeftConfig

peft_config = PrefixTuningConfig(
    task_type="SEQ_CLS",       
    inference_mode=False,      
    num_virtual_tokens=20,    
    prefix_projection=True,   
)

In [83]:
model = get_peft_model(model, peft_config)

In [84]:
# Посмотрим количество размороженных параметров для дообучения
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Всего параметров: {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,} ({trainable_params/total_params:.2%})")
print(f"Замороженных параметров: {frozen_params:,} ({frozen_params/total_params:.2%})")

Всего параметров: 124,267,014
Обучаемых параметров: 14,780,160 (11.89%)
Замороженных параметров: 109,486,854 (88.11%)


In [86]:
training_args = TrainingArguments(
    output_dir="../data/models/bert_base_prefix_tuning",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
)

In [87]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

/var/folders/gb/r8_rgfms4txcy1_nl4l107kc0000gn/T/ipykernel_49457/1970069337.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [89]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.565900,1.411587,0.528500,0.414806
2,1.292200,1.152619,0.588000,0.462975
3,1.119200,0.968914,0.694000,0.626955
4,0.965600,0.849734,0.753000,0.697516
5,0.877600,0.786606,0.782000,0.728256
6,0.824200,0.738928,0.796500,0.741942
7,0.794900,0.717464,0.800500,0.746372
8,0.775600,0.707262,0.800000,0.746098


TrainOutput(global_step=4000, training_loss=1.0268919067382813, metrics={'train_runtime': 1191.0821, 'train_samples_per_second': 107.465, 'train_steps_per_second': 3.358, 'total_flos': 3450430740114432.0, 'train_loss': 1.0268919067382813, 'epoch': 8.0})

- время дообучения: 19:50

- ресурсы: обучал на CPU

In [90]:
results = trainer.evaluate()

In [91]:
print(f"Loss: {results['eval_loss']:.4f}")
print(f"Accuracy: {results['eval_accuracy']:.4f}")
print(f"F1: {results['eval_f1']:.4f}")

Loss: 0.7073
Accuracy: 0.8000
F1: 0.7461


In [92]:
import torch
from sklearn.metrics import classification_report

test_dataloader = trainer.get_test_dataloader(tokenized_ds["test"])
predictions = []
labels = []

with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(trainer.model.device) for k, v in batch.items() if k != 'labels'}
        outputs = trainer.model(**batch)
        predictions.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
        
true_labels = tokenized_ds["test"]["label"]
print("\Отчет о классификации на test:")
print(classification_report(true_labels, predictions))

\Отчет о классификации на test:
              precision    recall  f1-score   support

           0       0.90      0.93      0.92       581
           1       0.78      0.97      0.86       695
           2       0.00      0.00      0.00       159
           3       0.86      0.84      0.85       275
           4       0.77      0.91      0.83       224
           5       0.00      0.00      0.00        66

    accuracy                           0.82      2000
   macro avg       0.55      0.61      0.58      2000
weighted avg       0.74      0.82      0.78      2000



/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/kulyaskin_mikhail/Library/Caches/pypoetry/virtualenvs/dl-nlp-7PEEyLUf-py3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 

# Таблица сравнение

| Подход            | Кол-во обучаемых параметров | Время дообучения | device | Accuracy | F1     |
| ----------------- | --------------------------- | ---------------- | ------ | -------- | ------ |
| без FT            | -                           | -                | -      | 0.2715   | 0.1188 |
| Full finetuning   | 109,486,854 (100%)          | 1:15:24          | GPU    | 0.9400   | 0.9399 |
| linear probing    | 529,670 (0.48%)             | 49:54            | CPU    | 0.4220   | 0.3219 |
| LoRA              | 446,982 (0.41%)             | 1:02:30          | GPU    | 0.8435   | 0.8413 |
| Prefix finetuning | 14,780,160 (11.89%)         | 19:50            | CPU    | 0.8000   | 0.7461 |

- в целом, размер модели относительно небольшой, поэтому есть смысл делать полный fine-tuning модели
- В номинации цена-качетсво тут побеждает Prefix finetuning, 20 минут на CPU дает 80% точности 